# 06 — Évaluation sur le test — verdict, erreurs classées et rapport

**Projet** : Extraction d'entités nommées par spaCy — le tagger appris, corroboré par des règles

## Objectifs pédagogiques

1. Ouvrir le split de test **une seule fois** et lire le verdict contractuel.
1. Ventiler la performance par type, par style et par canal, et comparer les surfaces **réservées** à celles vues à l'entraînement.
1. Classer les erreurs (inventée, bornes, manquée) et en montrer un exemple de chaque.
1. Écrire le rapport du projet, ses tables et ses figures, puis en tirer des recommandations chiffrées.

## 1. Mise en place

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, display

# --- Racine du projet ---------------------------------------------------------------------------
# Le notebook s'exécute depuis `notebooks/` : on remonte d'un cran pour pouvoir importer `src`.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hydra import compose, initialize_config_dir  # noqa: E402
from hydra.core.global_hydra import GlobalHydra  # noqa: E402

from src.data.schemas import (  # noqa: E402
    ENTITY_LABELS,
)
from src.schemas.config import validate_config  # noqa: E402
from src.utils.config_access import node  # noqa: E402
from src.utils.logging import setup_logging  # noqa: E402
from src.utils.paths import ProjectPaths  # noqa: E402

# --- Réglages d'affichage -----------------------------------------------------------------------
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 170)
# Le projet journalise au niveau INFO ; un notebook noierait ses tableaux sous ces lignes.
setup_logging(level="WARNING")

# --- Configuration : exactement celle de `python -m src.main` ------------------------------------
# Les notebooks travaillent sur un corpus réduit (320 messages) et un budget d'époques
# réduit : l'exécution reste rapide, et les valeurs absolues d'un notebook ne sont pas celles de la
# référence publiée par le README.
NB_DOCUMENTS = 320
NB_EPOCHS = 3

GlobalHydra.instance().clear()
with initialize_config_dir(config_dir=str(PROJECT_ROOT / "conf"), version_base=None):
    CONFIG = validate_config(
        compose(
            config_name="config",
            overrides=[
                "mode=train",
                f"data.n_samples={NB_DOCUMENTS}",
                f"train.epochs={NB_EPOCHS}",
                "seed=42",
                "log_level=WARNING",
                "train.callbacks.progress_bar=false",
                "train.callbacks.logging_every=100",
            ],
        )
    )

MODEL_NODE = dict(node(CONFIG, "model"))
TEXT_COLUMN = str(MODEL_NODE.get("text_column", "text"))
TARGET_COLUMN = str(MODEL_NODE.get("target", "label"))
ID_COLUMN = str(CONFIG.data.id_column or "msg_id")
SPLIT_COLUMN = str(CONFIG.data.group_column or "split")

# Les notebooks lisent **et écrivent** dans leur propre bac à sable : le corpus réduit et les
# artefacts des notebooks ne touchent ni `data/raw` ni `artifacts/`, donc ceux de `make train`
# restent ceux de la référence.
NB_PATHS = ProjectPaths(
    root=PROJECT_ROOT,
    data_dir=PROJECT_ROOT / "outputs" / "notebooks" / "data",
    artifacts_dir=PROJECT_ROOT / "outputs" / "notebooks" / "artifacts",
).ensure()

print(f"Projet            : {CONFIG.project.name}")
print(f"Tâche             : {CONFIG.metrics.task}")
print(f"Métrique primaire : {CONFIG.metrics.primary} (seuil contractuel : 0.7)")
print(f"Algorithme servi  : {CONFIG.model.algorithm} ({CONFIG.model.name})")
print(f"Types d'entités   : {len(ENTITY_LABELS)} — {', '.join(ENTITY_LABELS)}")
print(f"Bac à sable       : {NB_PATHS.artifacts_dir.relative_to(PROJECT_ROOT)}")

In [ ]:
from src.data.generators import SyntheticEntityCorpusGenerator
from src.data.loaders import EntityCorpusLoader

LOADER = EntityCorpusLoader(
    NB_PATHS,
    dataset_name=str(CONFIG.data.dataset_name),
    formats=tuple(CONFIG.data.formats),
    validation_enabled=bool(CONFIG.data.validation.raw),
    lazy_validation=bool(CONFIG.data.validation.lazy),
)

if LOADER.documents_path.exists():
    DOCUMENTS, SPANS = LOADER.load_corpus()
    METADATA = LOADER.load_metadata()
    print(f"Corpus lu depuis le bac à sable : {len(DOCUMENTS)} messages, graine {METADATA['seed']}")
else:
    # Un clone frais n'a pas encore de données : le notebook reste exécutable. Le corpus réduit est
    # persisté **dans le bac à sable** pour que les notebooks suivants et les pipelines travaillent
    # sur exactement les mêmes fichiers.
    bundle = SyntheticEntityCorpusGenerator.from_config(CONFIG.data).generate()
    DOCUMENTS, SPANS, METADATA = bundle.documents, bundle.queries, dict(bundle.metadata)
    LOADER.save_documents(DOCUMENTS)
    LOADER.save_annotations(SPANS)
    LOADER.save_metadata(METADATA)
    print(f"Bac à sable vide : corpus de {len(DOCUMENTS)} messages généré et persisté")

DOCUMENTS = DOCUMENTS.sort_values(ID_COLUMN).reset_index(drop=True)
SPANS = SPANS.sort_values([ID_COLUMN, "start", "end"]).reset_index(drop=True)

TRAIN_DOCUMENTS = LOADER.split("train", frame=DOCUMENTS)
VAL_DOCUMENTS = LOADER.split("val", frame=DOCUMENTS)
CALIBRATION_DOCUMENTS = LOADER.split("calibration", frame=DOCUMENTS)
TEST_DOCUMENTS = LOADER.split("test", frame=DOCUMENTS)
TRAIN_SPANS = LOADER.split_annotations("train", documents=DOCUMENTS, spans=SPANS)
VAL_SPANS = LOADER.split_annotations("val", documents=DOCUMENTS, spans=SPANS)
TEST_SPANS = LOADER.split_annotations("test", documents=DOCUMENTS, spans=SPANS)

print(
    "messages par split :",
    {
        name: len(frame)
        for name, frame in (
            ("train", TRAIN_DOCUMENTS),
            ("val", VAL_DOCUMENTS),
            ("calibration", CALIBRATION_DOCUMENTS),
            ("test", TEST_DOCUMENTS),
        )
    },
)
print(
    "mentions par split :",
    {
        name: len(frame)
        for name, frame in (("train", TRAIN_SPANS), ("val", VAL_SPANS), ("test", TEST_SPANS))
    },
)

## 2. Entraîner le modèle à évaluer

Le test ne s'ouvre qu'après l'entraînement : le modèle est ajusté sur `train`, surveillé sur `val`, et le split de test sert à **juger**, une fois.

In [ ]:
from src.models import build_model

MODEL = build_model(CONFIG)
FIT = MODEL.fit(TRAIN_DOCUMENTS, TRAIN_SPANS)
print(f"modèle : {MODEL.summary()}")
print(f"ajustement : {FIT.n_samples} messages, {FIT.epochs} époque(s), {FIT.duration_seconds:.2f}s")
print(
    "métriques d'entraînement (extrait) :",
    {
        key: round(value, 4)
        for key, value in FIT.metrics.items()
        if key in {"entity_f1", "entity_recall"}
    },
)

## 3. Le verdict contractuel

Le verdict compare la métrique principale à son seuil déclaré (`metrics.min_primary`). Il est **indéterminé** quand la métrique manque : un contrat qu'on ne peut pas lire n'est pas un contrat satisfait.

In [ ]:
from src.evaluation.evaluator import EntityEvaluator

EVALUATOR = EntityEvaluator(
    MODEL,
    config=CONFIG.model_dump(),
    paths=NB_PATHS,
    primary_metric=CONFIG.metrics.primary,
    min_primary_metric=CONFIG.metrics.min_primary,
)
RESULT = EVALUATOR.evaluate(TEST_DOCUMENTS, TEST_SPANS)

print(f"messages évalués : {RESULT.n_documents} | entités de référence : {RESULT.n_entities}")
print(
    f"métrique principale : {RESULT.primary_metric} = "
    f"{RESULT.metrics[RESULT.primary_metric]:.4f} (seuil : {RESULT.threshold})"
)
print(f"verdict : {RESULT.verdict} — {RESULT.verdict_detail.get('message', '')}")
display(pd.Series(RESULT.metrics).sort_values(ascending=False).to_frame("valeur").round(4))

**Ce qu'il faut retenir**

- La métrique principale est **micro** : toutes les mentions comptent pareil. La F1 macro, publiée à côté, compte les types à égalité — les deux lectures sont nécessaires.
- `partial_f1` compte une mention du bon type qui **recouvre** l'entité : l'écart avec `entity_f1` est le prix de l'exigence de bornes.
- Le verdict porte sur la validation du contrat de service, pas sur la qualité scientifique du modèle.

## 4. Par type, par segment, par surface

Une moyenne cache toujours la même chose : **quelqu'un** perd. Trois ventilations rendent la perte visible — le type, le segment d'écriture, et la provenance de la surface.

In [ ]:
display(RESULT.per_label.round(4))
FRAGILE = RESULT.per_label[RESULT.per_label["label"] != "micro"].sort_values("f1").iloc[0]
print(
    f"type le plus fragile : {FRAGILE['label']} (F1 {FRAGILE['f1']:.4f}, "
    f"{int(FRAGILE['support'])} mentions de référence)"
)

In [ ]:
display(RESULT.segments.round(4))
print("segments publiés :", sorted(set(RESULT.segments["segment"])))
display(RESULT.holdout.round(4))
print(f"rappel sur les surfaces réservées   : {RESULT.metrics['holdout_recall']:.4f}")
print(f"rappel sur les surfaces vues au train : {RESULT.metrics['seen_surface_recall']:.4f}")
print(f"écart (ce que la mémorisation explique) : {RESULT.metrics['holdout_gap']:.4f}")

**Ce qu'il faut retenir**

- La ventilation par **style** est la plus instructive du corpus : les messages abrégés écrivent les mêmes entités autrement.
- L'écart « surface réservée / surface vue » est la mesure honnête de ce que le modèle a **appris** plutôt que mémorisé.
- Un type fragile avec peu de support se lit avec prudence : trois mentions ne font pas une conclusion.

## 5. Les trois familles d'erreurs

Le projet classe les erreurs plutôt que de les compter : une mention **inventée** coûte de la précision, une mention **manquée** du rappel, et une erreur de **bornes** est un cas à part — le type est bon, la découpe ne l'est pas.

In [ ]:
ERREURS = RESULT.errors
print("erreurs publiées :", len(ERREURS))
if not ERREURS.empty:
    display(ERREURS["kind"].value_counts().to_frame("erreurs"))
    display(ERREURS.head(8)[["msg_id", "kind", "label", "surface", "predicted_surface", "context"]])

In [ ]:
if not ERREURS.empty:
    for kind in ("inventee", "bornes", "manquee"):
        exemple = ERREURS[ERREURS["kind"] == kind]
        if exemple.empty:
            print(f"{kind:10s} : aucune erreur de ce type sur le test")
            continue
        row = exemple.iloc[0]
        print(
            f"{kind:10s} : {row['label']} {str(row['surface'])!r} "
            f"(prédit : {row['predicted_surface']!r})"
        )

**Ce qu'il faut retenir**

- Les erreurs de **bornes** sont les plus coûteuses en apparence : elles comptent double (faux positif et faux négatif) et le `partial_f1` montre qu'elles recouvrent souvent la bonne entité.
- Une mention **inventée** sur un distracteur déclaré (une ville, une référence de facture) est une erreur prévisible : le corpus la contenait pour être mesurée.
- Le contexte est publié avec chaque erreur : un lecteur peut juger sans rouvrir le corpus.

## 6. Les références : plancher trivial et règles seules

Un score ne se lit pas seul. Deux références sont publiées : le plancher trivial (un système qui n'annote rien) et la couche de règles **apprise sur le train**, mesurée sur le même test.

In [ ]:
from src.training.metrics import trivial_floor

BASELINES = pd.DataFrame(RESULT.baselines).T
display(BASELINES.round(4))
print("plancher trivial :", trivial_floor())
reference = float(RESULT.baselines.get("regles", {}).get("entity_f1", float("nan")))
gain = RESULT.metrics["entity_f1"] - reference
print(f"gain du modèle sur les règles seules : {gain:+.4f} de F1 micro")

**Ce qu'il faut retenir**

- Le plancher trivial vaut 0,0 par convention : un système qui n'annonce rien n'a aucune précision.
- La référence « règles » est apprise sur le **train** et mesurée sur le test : c'est la même discipline que pour le modèle.
- Un gain plus petit que l'écart sur les surfaces réservées serait un signal clair : le modèle n'apprendrait pas grand-chose de plus que la liste.

## 7. Le rapport écrit par le projet

L'évaluation produit un rapport Markdown, ses tables CSV et ses figures. C'est l'artefact que lit un lecteur pressé — et celui qui explique chaque chiffre.

In [ ]:
from src.evaluation.reports import ReportBuilder

BUNDLE = ReportBuilder(CONFIG.model_dump(), paths=NB_PATHS).build(
    RESULT,
    model_summary={
        "name": MODEL.name,
        "framework": MODEL.framework,
        "algorithm": MODEL.algorithm,
        "task": MODEL.task,
        "labels": MODEL.labels,
        "state": MODEL.state,
    },
    metadata=METADATA,
    documents=DOCUMENTS,
    spans=SPANS,
)
print("verdict du rapport :", BUNDLE.verdict)
print(BUNDLE.summary)
for path in BUNDLE.artifacts:
    relative = Path(path).relative_to(NB_PATHS.artifacts_dir)
    print(f"  - {relative}")

In [ ]:
LINES = BUNDLE.report_path.read_text(encoding="utf-8").splitlines()
print("\n".join(LINES[:42]))

In [ ]:
from src.visualization.plots import plot_holdout, plot_per_label

figure_types = plot_per_label(RESULT.per_label, NB_PATHS.figures_dir / "test_per_label.png")
figure_reserves = plot_holdout(RESULT.holdout, NB_PATHS.figures_dir / "test_holdout.png")
display(Image(filename=str(figure_types)))
display(Image(filename=str(figure_reserves)))

**Ce qu'il faut retenir**

- Un rapport qui se contente d'un score n'est pas un rapport : celui-ci publie les tables, les figures et les erreurs, chacun dans un fichier.
- Les artefacts du notebook vont dans `outputs/notebooks/` : le rapport de référence reste celui de `make evaluate`.
- Le rapport est régénéré à chaque `mode=evaluate` : il ne peut pas mentir sur l'état du modèle servi.

## 8. Recommandations chiffrées

Ce que ce notebook permet de décider, avec les chiffres qui le justifient.

In [ ]:
F1 = RESULT.metrics["entity_f1"]
MACRO = RESULT.metrics["macro_f1"]
BORNES = RESULT.metrics["boundary_accuracy"]
PARTIEL = RESULT.metrics["partial_f1"]
ECART = RESULT.metrics["holdout_gap"]
RAPPEL_RESERVE = RESULT.metrics["holdout_recall"]

print("1. Qualité globale")
print(f"   F1 micro {F1:.4f} / macro {MACRO:.4f} : la hiérarchie des types n'est pas plate.")
print(
    f"   bornes exactes {BORNES:.4f} contre F1 partielle {PARTIEL:.4f} : "
    f"{PARTIEL - BORNES:.4f} de mentions bien typées mais mal découpées."
)
print()
print("2. Ce qui vient de la liste et ce qui vient du modèle")
print(
    f"   rappel sur surfaces vues {RESULT.metrics['seen_surface_recall']:.4f} contre "
    f"{RAPPEL_RESERVE:.4f} sur les réservées (écart {ECART:.4f})."
)
print(
    f"   les {int(RESULT.holdout['n_entities'].sum())} mentions réservées sont la vraie mesure "
    "de la généralisation."
)
print()
print("3. Où agir")
print(
    f"   type le plus fragile : {FRAGILE['label']} (F1 {FRAGILE['f1']:.4f}) — enrichir ses "
    "surfaces à l'entraînement, pas ses motifs."
)
segments = RESULT.segments.set_index(["segment", "value"])
if ("style", "abrege") in segments.index:
    print(
        f"   style abrégé : F1 {segments.loc[('style', 'abrege'), 'entity_f1']:.4f} — vérifier "
        "les variantes d'écriture des motifs."
    )
print(
    f"   verdict contractuel : {RESULT.verdict} "
    f"(seuil {RESULT.threshold} sur {RESULT.primary_metric})."
)

**Ce qu'il faut retenir**

- Trois décisions, trois chiffres : enrichir les surfaces des types fragiles, surveiller l'écart sur les surfaces réservées, garder le seuil contractuel comme porte de déploiement.
- Un modèle qui gagne sur les surfaces vues et perd sur les réservées mémorise : le tableau des références est là pour le dire.
- Le test a été ouvert une fois, dans ce notebook : toute nouvelle décision devra se prendre sur `val` et se vérifier sur un nouveau test.

## Synthèse

| Question | Réponse du notebook | Artefact |
| --- | --- | --- |
| Le contrat est-il tenu ? | `RESULT.verdict`, `threshold` | `evaluation_report.md` |
| Quel type fragile ? | `RESULT.per_label` | `per_label.csv` |
| Quel segment perd ? | `RESULT.segments` | `segment_metrics.csv` |
| Le modèle mémorise-t-il ? | `holdout_recall`, `holdout_gap` | `holdout.csv` |
| Quelles erreurs corriger ? | `RESULT.errors` | `errors.csv` |

**Fin du parcours** : de la carte du corpus (01) au rapport (06). Reprendre la configuration, pas le code : `conf/`, puis `python -m src.main mode=all`.